In [3]:
mkdir -p data/

In [4]:
!aws s3 cp "s3://de300-hw3-nyctlc-549787090008-us-east-1-an/yellow_tripdata_2026-01.parquet" data/
!aws s3 cp "s3://de300-hw3-nyctlc-549787090008-us-east-1-an/green_tripdata_2026-01.parquet" data/

download: s3://de300-hw3-nyctlc-549787090008-us-east-1-an/yellow_tripdata_2026-01.parquet to data/yellow_tripdata_2026-01.parquet
download: s3://de300-hw3-nyctlc-549787090008-us-east-1-an/green_tripdata_2026-01.parquet to data/green_tripdata_2026-01.parquet


### Part 1: EC2 and Spark setup

In [ ]:
!sudo yum update
!sudo yum install -y openjdk-11-jdk python3-pip awscli
!pip3 install pyspark pandas pyarrow

### Part 2:  Start Spark

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC Taxi Analytics Assignment")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 22:04:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Part 3: Load yellow and green taxi data

In [6]:
yellow_path = "data/yellow_tripdata_2026-01.parquet"
green_path = "data/green_tripdata_2026-01.parquet"

yellow_raw = spark.read.parquet(yellow_path)
green_raw = spark.read.parquet(green_path)


In [7]:
yellow_raw.printSchema()
green_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: ti

In [8]:
yellow_raw.show(5, truncate=False)
green_raw.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2026-01-01 00:54:04 |2026-01-01 00:59:37  |1              |0.97         |1         |N                 |239         |238 

### Part 4: Standardize schemas

Yellow and green taxi datasets use similar but not always identical column names. You should standardize the key columns (i.e., date times and location IDs) before combining the datasets.

In [9]:
from pyspark.sql import functions as F

yellow_raw = yellow_raw.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
                       .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
                       .withColumn("taxi_type", F.lit("yellow"))

green_raw = green_raw.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
                     .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime") \
                     .withColumn("taxi_type", F.lit("green"))


combined_df = yellow_raw.unionByName(green_raw, allowMissingColumns=True)

In [10]:
combined_df.show(5)

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------+---------+---------+
|VendorID|    pickup_datetime|   dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|taxi_type|ehail_fee|trip_type|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------+---------+---------+
|       2|2026-01-01 00:54:04|2026-01-01 00:59:37

### Part 5: Clean the data

You should remove invalid or suspicious records.

Required cleaning rules:

Remove rows with missing pickup or drop-off timestamps.
Remove trips with non-positive distance.
Remove trips with negative fares.
Remove trips with negative total amount.
Remove trips longer than 24 hours.
Remove trips with drop-off time before pickup time.

In [11]:
cleaned_df = (combined_df
    .filter(F.col("pickup_datetime").isNotNull() & F.col("dropoff_datetime").isNotNull())
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .filter((F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) <= 86400)
    .filter(F.col("dropoff_datetime") > F.col("pickup_datetime"))
)

In [12]:
print(f"Before cleaning: {combined_df.count():,}")
print(f"After cleaning:  {cleaned_df.count():,}")
print(f"Rows removed:    {combined_df.count() - cleaned_df.count():,}")

Before cleaning: 3,765,161


After cleaning:  3,557,037
Rows removed:    208,124


### Part 6: Required Analytics

Question 1: Which taxi type had more trips?

In [13]:
q1 = cleaned_df.groupBy("taxi_type").count().orderBy(F.col("count").desc())
q1.show()

+---------+-------+
|taxi_type|  count|
+---------+-------+
|   yellow|3518128|
|    green|  38909|
+---------+-------+



Question 2: What was the average fare by taxi type?

In [14]:
q2 = cleaned_df.groupBy("taxi_type").agg(F.avg("fare_amount").alias("avg_fare"))
q2.show()

+---------+------------------+
|taxi_type|          avg_fare|
+---------+------------------+
|   yellow| 21.08260282173478|
|    green|16.003356806908457|
+---------+------------------+



Question 3: What was the average trip distance by taxi type?

In [15]:
q3 = cleaned_df.groupBy("taxi_type").agg(F.avg("trip_distance").alias("avg_distance"))
q3.show()

+---------+-----------------+
|taxi_type|     avg_distance|
+---------+-----------------+
|   yellow|6.757001624157525|
|    green|12.98990593435966|
+---------+-----------------+



Question 5: What percentage of trips were under 2 miles?

In [16]:
total = cleaned_df.count()
under_2 = cleaned_df.filter(F.col("trip_distance") < 2).count()
pct = (under_2 / total) * 100
print(f"Trips under 2 miles: {pct:.2f}%")

Trips under 2 miles: 51.95%


Question 8: Predict the fare amount using non-fare predictor columns.

Construct a regression model (random forest, linear regression, or gradient boosted trees) in PySpark. You should only use the non-fare columns as predictors. Use 80% of the data to train, and the remaining 20% to test the model.

Report both training and testing RMSE. Provide an analysis on the importance (or relevance) of the predictors.

Plot the predicted fare amount vs the actual fare amount. Save it as a .png and write to your S3 bucket.

In [17]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)


In [18]:
indexer = StringIndexer(inputCol="taxi_type", outputCol="taxi_type_index", handleInvalid="keep")

feature_cols = [
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "RatecodeID",
    "payment_type",
    "congestion_surcharge",
    "taxi_type_index",
]

label_col = "fare_amount"

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip",
)

In [19]:
model_df = cleaned_df.select(
    "trip_distance", "PULocationID", "DOLocationID",
    "passenger_count", "RatecodeID", "payment_type",
    "congestion_surcharge", "taxi_type", "fare_amount"
).dropna()

In [20]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count():,}")
print(f"Test rows:     {test_df.count():,}")


Training rows: 2,045,084


Test rows:     511,300


In [21]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    numTrees=80,
    maxDepth=8,
    seed=42,
)

pipeline = Pipeline(stages=[indexer, assembler, rf])
model = pipeline.fit(train_df)

26/05/23 22:06:35 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 140.8 MiB so far)
26/05/23 22:06:35 WARN BlockManager: Persisting block rdd_186_0 to disk instead.
26/05/23 22:06:52 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:07:06 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:07:23 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:07:49 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:08:15 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:08:49 WARN MemoryStore: Not enough space to cache rdd_186_0 in memory! (computed 211.3 MiB so far)
26/05/23 22:09:30 WARN DAGScheduler: Broadcasting large task binary with size 1596.3 KiB
26/05/23 22:09:32 WARN MemoryStore: Not enough space 

In [22]:
rmse_evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
r2_evaluator   = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")
mae_evaluator  = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae")

train_preds = model.transform(train_df)
test_preds  = model.transform(test_df)

train_rmse = rmse_evaluator.evaluate(train_preds)
test_rmse  = rmse_evaluator.evaluate(test_preds)
test_r2    = r2_evaluator.evaluate(test_preds)
test_mae   = mae_evaluator.evaluate(test_preds)

print(f"\nTraining RMSE: {train_rmse:.4f}")
print(f"Testing RMSE:  {test_rmse:.4f}")
print(f"Testing MAE:   {test_mae:.4f}")
print(f"Testing R^2:   {test_r2:.4f}")



Training RMSE: 5.8434
Testing RMSE:  6.1287
Testing MAE:   2.8301
Testing R^2:   0.8895


In [23]:
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

importance_pdf = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=True)
)

print("Feature importances:")
print(importance_pdf.sort_values("importance", ascending=False).to_string(index=False))


Feature importances:
             feature  importance
       trip_distance    0.577005
          RatecodeID    0.267131
congestion_surcharge    0.063800
        PULocationID    0.050917
        DOLocationID    0.033616
     passenger_count    0.003457
     taxi_type_index    0.002240
        payment_type    0.001835


In [24]:
plot_pdf = (
    test_preds
    .select("fare_amount", "prediction")
    .sample(fraction=0.25, seed=7)
    .limit(1000)
    .toPandas()
)

plt.figure(figsize=(8, 6))
plt.scatter(plot_pdf["fare_amount"], plot_pdf["prediction"], alpha=0.35)

min_val = min(plot_pdf["fare_amount"].min(), plot_pdf["prediction"].min())
max_val = max(plot_pdf["fare_amount"].max(), plot_pdf["prediction"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red", label="Perfect prediction")

plt.xlabel("Actual Fare Amount")
plt.ylabel("Predicted Fare Amount")
plt.title("Random Forest: Actual vs. Predicted Fare Amount")
plt.legend()
plt.tight_layout()

actual_vs_predicted_path = output_dir / "actual_vs_predicted.png"
plt.savefig(actual_vs_predicted_path, dpi=150)
plt.close()


In [25]:
plt.figure(figsize=(8, 5))
plt.barh(importance_pdf["feature"], importance_pdf["importance"])
plt.xlabel("Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()

feature_importance_path = output_dir / "feature_importances.png"
plt.savefig(feature_importance_path, dpi=150)
plt.close()

In [28]:
import boto3
BUCKET_NAME = "phiri-rhema-lab3"
s3 = boto3.client("s3")

actual_vs_predicted_path = Path("outputs/actual_vs_predicted.png")
feature_importance_path = Path("outputs/feature_importances.png")

for path in [actual_vs_predicted_path, feature_importance_path]:
    s3.upload_file(str(path), BUCKET_NAME, f"outputs/{path.name}")
    print(f"Uploaded to S3: outputs/{path.name}")

Uploaded to S3: outputs/actual_vs_predicted.png
Uploaded to S3: outputs/feature_importances.png


### Part 7: Write Results to S3

In [ ]:
import boto3
from pathlib import Path

BUCKET_NAME = "phiri-rhema-lab3"
s3 = boto3.client("s3")

q1.write.mode("overwrite").parquet("outputs/trips_by_type")
q2.write.mode("overwrite").parquet("outputs/avg_fare_by_type")

import os
for folder in ["outputs/trips_by_type", "outputs/avg_fare_by_type"]:
    for filename in os.listdir(folder):
        local_path = f"{folder}/{filename}"
        s3_key = f"nyc-taxi-assignment/{folder}/{filename}"
        s3.upload_file(local_path, BUCKET_NAME, s3_key)